# MEPS Part II: Modeling Total Health Care Expenditures

The Part II regression task predicts how much an adult spent on health care in calendar year 2023 from the MEPS HC-251 Full Year Consolidated file.

The target is `TOTEXP23`. It is a non-negative dollar amount summed across all event types: office-based visits, outpatient, emergency room, inpatient, home health, dental, vision, prescription drugs, and other. A large share of adults have `TOTEXP23 = 0` and the remaining values are heavily right-skewed by a small number of very high-cost cases.

The modeling goal is to compare interpretable and higher-performing regression models while keeping the feature choices tied to the codebook, cleaning decisions, and project question.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## File paths


In [ ]:
# Colab option
from google.colab import drive
drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/med_project")

# Local/PyCharm option (Commented out for Colab)
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"

DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)
print("Reports folder:", REPORTS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/med_project
Raw data folder: /content/drive/MyDrive/med_project/data/raw
Reports folder: /content/drive/MyDrive/med_project/reports


## Load the MEPS model-ready files

The cleaning notebook produced two adult-only model-ready files. The primary file is the headline modeling input. The sensitivity file adds the SAQ mental-health variables (`K6SUM42`, `PHQ242`, `ADINTR42`, `ADDPRS42`) and the SAQ-derived `mental_health_screen_count`. Both files share the same row order, so the train/test split below applies to both consistently.

In [ ]:
primary_path = DATA_PROCESSED / "meps_model_ready_v1.csv"
sensitivity_path = DATA_PROCESSED / "meps_model_ready_v1_with_saq.csv"

meps = pd.read_csv(primary_path, low_memory=False)
meps_with_saq = pd.read_csv(sensitivity_path, low_memory=False)

print("Primary model-ready shape:    ", meps.shape)
print("Sensitivity (with SAQ) shape: ", meps_with_saq.shape)
meps.head()

Primary model-ready shape:     (15123, 51)
Sensitivity (with SAQ) shape:  (15123, 56)


,TOTEXP23,TOTEXP23_log1p,AGE23X,SEX,RACETHX,HISPANX,MARRY23X,REGION23,EDUCYR,HIDEG,...,DLAYCA42,AFRDCA42,DLAYPM42,AFRDPM42,DLAYDN42,AFRDDN42,chronic_condition_count,limitation_indicator_count,physical_difficulty_count,access_barrier_count
0,646,6.472346,58,2,3,2,3.0,2,17.0,5.0,...,2.0,2.0,2.0,2.0,1.0,1.0,1,0,0,2
1,1894,7.546974,27,1,3,2,5.0,2,12.0,3.0,...,2.0,2.0,2.0,2.0,2.0,2.0,1,0,0,0
2,986,6.894670,49,2,2,2,5.0,2,17.0,5.0,...,2.0,2.0,2.0,2.0,2.0,2.0,0,0,0,0
3,1312,7.180070,75,2,2,2,2.0,2,12.0,3.0,...,2.0,2.0,2.0,2.0,1.0,1.0,4,0,0,2
4,0,0.000000,23,1,2,2,5.0,2,11.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,0,0,0,0


## MEPS survey design and target distribution

MEPS is not a simple random sample. It is a nationally representative survey of the civilian noninstitutionalized U.S. population. The 2023 design uses a complex sampling design, and the public-use file includes survey design variables for weighted estimation and variance estimation.

For the Full Year Consolidated file, `PERWT23F` is the final 2023 person-level weight. `VARSTR` is the variance estimation stratum and `VARPSU` is the variance estimation primary sampling unit. These variables are retained for descriptive checks and documentation, but they are not used as model predictors.

The machine learning train/test split is also stratified, but that is a separate modeling step. In the modeling split, stratification only means the target distribution is kept similar in the training and test sets — for the regression target this is done by quintiling `TOTEXP23` so high-cost cases are represented in both sets.

The target is heavily right-skewed with a spike at zero, so a single accuracy-style metric is not enough. Evaluation includes MAE, RMSE, R², a high-cost segment review, and a `log1p(TOTEXP23)` sensitivity version trained alongside the raw target.

In [ ]:
target_col = "TOTEXP23"
target_log_col = "TOTEXP23_log1p"

n = len(meps)
n_zero = int((meps[target_col] == 0).sum())

target_summary = pd.DataFrame({
    "measure": [
        "adults",
        "zero spenders",
        "zero share",
        "mean",
        "median",
        "p90",
        "p95",
        "p99",
        "raw skewness",
        "log1p skewness",
    ],
    "value": [
        n,
        n_zero,
        n_zero / n,
        meps[target_col].mean(),
        meps[target_col].median(),
        meps[target_col].quantile(0.90),
        meps[target_col].quantile(0.95),
        meps[target_col].quantile(0.99),
        meps[target_col].skew(),
        meps[target_log_col].skew(),
    ],
})

target_summary

,measure,value
0,adults,15123.000000
1,zero spenders,2053.000000
2,zero share,0.135753
3,mean,9634.878728
4,median,2458.000000
5,p90,24243.200000
6,p95,42134.200000
7,p99,105948.020000
8,raw skewness,7.617585
9,log1p skewness,-1.128287


In [ ]:
design_cols = ["PERWT23F", "VARSTR", "VARPSU"]

for col in design_cols:
    if col in meps.columns:
        print(f"{col}: present")
    else:
        print(f"{col}: not in primary model-ready file")

if {"VARSTR", "VARPSU"}.issubset(meps.columns):
    print("\nSurvey design check:")
    print("Number of variance strata:", meps["VARSTR"].nunique())
    print("Number of variance PSUs:  ", meps["VARPSU"].nunique())

if "PERWT23F" in meps.columns:
    print("\nPerson-level weight check:")
    print(meps["PERWT23F"].describe())
else:
    print("\nNote: PERWT23F was excluded from the primary model-ready file by design.")
    print("It is available in the interim cleaned file for weighted descriptive checks.")

PERWT23F: not in primary model-ready file
VARSTR: not in primary model-ready file
VARPSU: not in primary model-ready file

Note: PERWT23F was excluded from the primary model-ready file by design.
It is available in the interim cleaned file for weighted descriptive checks.


## Weighted target check

If the person-level weight `PERWT23F` is available in the loaded file, we compare the unweighted and weighted mean of `TOTEXP23`. The cleaning notebook drops `PERWT23F` from the model-ready file by design (weights are kept in the interim file for survey-aware checks), so this cell may report that the weight is unavailable. That is expected and not a problem for modeling.

In [ ]:
if "PERWT23F" in meps.columns:
    unweighted_mean = meps[target_col].mean()
    weighted_mean = np.average(meps[target_col], weights=meps["PERWT23F"])

    target_weight_check = pd.DataFrame({
        "measure": [
            "Unweighted mean TOTEXP23",
            "Weighted mean TOTEXP23 using PERWT23F",
        ],
        "value": [unweighted_mean, weighted_mean],
    })

    print(target_weight_check)
else:
    interim_path = DATA_INTERIM / "meps_first_pass_cleaned_candidates.csv"
    if interim_path.exists():
        interim = pd.read_csv(interim_path, low_memory=False)
        if "PERWT23F" in interim.columns and target_col in interim.columns:
            unweighted_mean = interim[target_col].mean()
            weighted_mean = np.average(
                interim[target_col],
                weights=interim["PERWT23F"],
            )
            target_weight_check = pd.DataFrame({
                "measure": [
                    "Unweighted mean TOTEXP23 (interim file)",
                    "Weighted mean TOTEXP23 using PERWT23F (interim file)",
                ],
                "value": [unweighted_mean, weighted_mean],
            })
            print(target_weight_check)
        else:
            print("PERWT23F is not available for the weighted target check.")
    else:
        print("PERWT23F is not available for the weighted target check.")

                                             measure        value
0            Unweighted mean TOTEXP23 (interim file)  9634.878728
1  Weighted mean TOTEXP23 using PERWT23F (interim...  8578.532337


## Poverty and feature engineering

Poverty is treated as a category in the main model. `POVCAT23` is the primary poverty feature: it is the public-use 5-level category produced by the MEPS team (poor, near poor, low income, middle income, high income). The continuous percent-of-poverty-line variable `POVLEV23` was kept in the interim file as a sensitivity option but was dropped from the primary model-ready file to avoid collinearity.

The first feature engineering pass was completed in the cleaning notebook and is already present in the loaded file as five summary features tied to the project question:

- chronic condition count
- limitation indicator count (yes/no general limitation screeners)
- physical difficulty count (Census six-item serious-difficulty set)
- access barrier count (cost-related delays / unaffordability across medical, prescription, dental)
- mental health screen count (SAQ-only, lives in the sensitivity file)

These summaries reduce redundancy by replacing several related raw variables with clearer grouped features. The cell below verifies that the engineered features are present and shows their distributions.

In [ ]:
meps_fe = meps.copy()

engineered_cols = [
    "POVCAT23",
    "chronic_condition_count",
    "limitation_indicator_count",
    "physical_difficulty_count",
    "access_barrier_count",
]

for col in engineered_cols:
    print("\n" + "=" * 80)
    print(col)

    if col not in meps_fe.columns:
        print("NOT PRESENT in primary file. Re-run the cleaning notebook.")
        continue

    if meps_fe[col].nunique(dropna=False) <= 20:
        print(meps_fe[col].value_counts(dropna=False).sort_index())
    else:
        print(meps_fe[col].describe())


POVCAT23
POVCAT23
1    2074
2     658
3    1866
4    4250
5    6275
Name: count, dtype: int64

chronic_condition_count
chronic_condition_count
0    5478
1    3253
2    2423
3    1842
4    1245
5     606
6     206
7      57
8      10
9       3
Name: count, dtype: int64

limitation_indicator_count
limitation_indicator_count
0    12057
1     1363
2      788
3      626
4      289
Name: count, dtype: int64

physical_difficulty_count
physical_difficulty_count
0    12035
1     1701
2      724
3      384
4      198
5       62
6       19
Name: count, dtype: int64

access_barrier_count
access_barrier_count
0    12241
1     1070
2     1052
3      328
4      220
5       83
6      129
Name: count, dtype: int64


## Predictor sets

Two model versions are created. The first version uses the primary adult features (no SAQ). The second version adds the SAQ mental-health features from the sensitivity file.

Related raw variables are not added when a clearer engineered count is already used. This keeps the first model easier to explain and reduces duplicate information. The chronic-condition flags, the per-domain limitation/difficulty flags, and the per-domain access-barrier flags are summarized by their respective counts.

In [ ]:
baseline_feature_cols = [
    # Demographics and location
    "AGE23X",
    "SEX",
    "RACETHX",
    "HISPANX",
    "MARRY23X",
    "REGION23",
    "EDUCYR",
    "HIDEG",
    "BORNUSA",
    "FAMSZEYR",

    # Poverty and socioeconomic context
    "POVCAT23",
    "FOODST23",

    # Health need
    "RTHLTH42",
    "MNHLTH42",
    "chronic_condition_count",

    # Functioning and disability
    "ANYLMI23",
    "limitation_indicator_count",
    "physical_difficulty_count",

    # Insurance and care access
    "INSC1231",
    "HAVEUS42",
    "OFFER31X",
    "HELD31X",

    # Broader affordability pressure
    "access_barrier_count",
]

saq_feature_cols = [
    "K6SUM42",
    "PHQ242",
    "mental_health_screen_count",
]

baseline_feature_cols = [col for col in baseline_feature_cols if col in meps_fe.columns]
saq_feature_cols = [col for col in saq_feature_cols if col in meps_with_saq.columns]

print("Baseline features:", len(baseline_feature_cols))
print(baseline_feature_cols)

print("\nSAQ features:", len(saq_feature_cols))
print(saq_feature_cols)

Baseline features: 23
['AGE23X', 'SEX', 'RACETHX', 'HISPANX', 'MARRY23X', 'REGION23', 'EDUCYR', 'HIDEG', 'BORNUSA', 'FAMSZEYR', 'POVCAT23', 'FOODST23', 'RTHLTH42', 'MNHLTH42', 'chronic_condition_count', 'ANYLMI23', 'limitation_indicator_count', 'physical_difficulty_count', 'INSC1231', 'HAVEUS42', 'OFFER31X', 'HELD31X', 'access_barrier_count']

SAQ features: 3
['K6SUM42', 'PHQ242', 'mental_health_screen_count']


## Excluded variables

Identifiers, survey design variables, the secondary poverty measure, and all dollar/utilization columns mechanically tied to `TOTEXP23` were already excluded by the cleaning notebook before the model-ready file was saved. The list below documents what is intentionally absent from the predictor space.

Weights and design variables are retained in the interim cleaned file for weighted descriptive checks and survey-aware sensitivity work, but they are not used as predictors.

In [ ]:
excluded_from_predictors = [
    # Identifiers and survey design (dropped during cleaning)
    "DUPERSID",
    "PANEL",
    "DATAYEAR",
    "PERWT23F",
    "VARSTR",
    "VARPSU",

    # Direct target leakage: every other 2023 dollar field except TOTEXP23 itself,
    # plus utilization counts. The cleaning notebook scans the full MEPS column space
    # for these and drops them by prefix/suffix rule.
    "TOTSLF23", "TOTMCR23", "TOTMCD23", "TOTPRV23", "TOTTCH23",
    "OBVEXP23", "OPVEXP23", "ERTEXP23", "IPTEXP23", "RXEXP23",
    "OBTOTV23", "OPTOTV23", "ERTOT23", "IPDIS23", "RXTOT23",

    # Secondary poverty measure (collinear with POVCAT23)
    "POVLEV23",

    # Structurally-missing for adults
    "WRKLIM31",

    # Raw variables summarized by engineered counts
    "HIBPDX", "CHOLDX", "DIABDX_M18", "ASTHDX", "ARTHDX",
    "CANCERDX", "CHDDX", "STRKDX", "EMPHDX",
    "WLKLIM31", "ACTLIM31", "SOCLIM31", "COGLIM31",
    "DFHEAR42", "DFSEE42", "DFCOG42", "DFWLKC42", "DFDRSB42", "DFERND42",
    "DLAYCA42", "AFRDCA42", "DLAYPM42", "AFRDPM42", "DLAYDN42", "AFRDDN42",

    # SAQ raw screen items summarized by mental_health_screen_count
    "ADINTR42", "ADDPRS42",
]

excluded_present_in_primary = [c for c in excluded_from_predictors if c in meps_fe.columns]
print("Excluded items still present in the primary file:", len(excluded_present_in_primary))
excluded_present_in_primary

Excluded items still present in the primary file: 25


['HIBPDX',
 'CHOLDX',
 'DIABDX_M18',
 'ASTHDX',
 'ARTHDX',
 'CANCERDX',
 'CHDDX',
 'STRKDX',
 'EMPHDX',
 'WLKLIM31',
 'ACTLIM31',
 'SOCLIM31',
 'COGLIM31',
 'DFHEAR42',
 'DFSEE42',
 'DFCOG42',
 'DFWLKC42',
 'DFDRSB42',
 'DFERND42',
 'DLAYCA42',
 'AFRDCA42',
 'DLAYPM42',
 'AFRDPM42',
 'DLAYDN42',
 'AFRDDN42']

## Build model versions

Version A tests the core adult features without SAQ. It is the headline configuration because every adult in the analytic sample contributes to the fit.

Version B adds the SAQ mental-health features so the SAQ layer can be evaluated instead of assumed to help. Because SAQ items are missing for adults outside the SAQ subframe, the modeling pipeline imputes those missing values inside the preprocessor; this means Version B does not restrict to SAQ-eligible adults but rather tests whether SAQ signals add predictive value when available.

In [ ]:
model_versions = {
    "A_baseline_adult": baseline_feature_cols,
    "B_baseline_plus_saq": baseline_feature_cols + saq_feature_cols,
}

for name, cols in model_versions.items():
    print(name, len(cols))

A_baseline_adult 23
B_baseline_plus_saq 26


## Train/test split

The train/test split is stratified by quintile of `TOTEXP23` so the heavy right tail is represented in both the training and test sets. This is a modeling step only. It does not replace the MEPS survey design variables or the person-level weight.

The split is done once on the sensitivity file (which contains every column that any version uses), so Version A and Version B see identical train and test rows. Only the column set differs between versions.

In [ ]:
model_df = meps_with_saq.dropna(subset=[target_col]).copy()
model_df[target_col] = model_df[target_col].astype(float)
model_df[target_log_col] = np.log1p(model_df[target_col])

target_bins = pd.qcut(model_df[target_col], q=5, labels=False, duplicates="drop")

if target_bins.value_counts(dropna=False).min() < 2 or target_bins.nunique() < 2:
    strat = None
    print("Using non-stratified split (insufficient bin size).")
else:
    strat = target_bins
    print("Using quintile-stratified split.")

train_df, test_df = train_test_split(
    model_df,
    test_size=0.25,
    random_state=42,
    stratify=strat,
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain target summary:")
print(train_df[target_col].describe().round(2))

print("\nTest target summary:")
print(test_df[target_col].describe().round(2))

# Define the high-cost threshold from the test set so it is held-out reference.
high_cost_threshold = float(np.quantile(test_df[target_col], 0.90))
high_cost_mask_test = test_df[target_col] >= high_cost_threshold
print(f"\nHigh-cost threshold (90th pct of test actuals): ${high_cost_threshold:,.0f}")
print(f"High-cost adults in test set: {int(high_cost_mask_test.sum())}")

Using quintile-stratified split.
Train shape: (11342, 56)
Test shape: (3781, 56)

Train target summary:
count     11342.00
mean       9680.36
std       23980.31
min           0.00
25%         401.00
50%        2461.00
75%        8719.25
max      574675.00
Name: TOTEXP23, dtype: float64

Test target summary:
count      3781.00
mean       9498.44
std       21141.93
min           0.00
25%         391.00
50%        2451.00
75%        8704.00
max      312186.00
Name: TOTEXP23, dtype: float64

High-cost threshold (90th pct of test actuals): $24,412
High-cost adults in test set: 379


## Preprocessing setup

Categorical variables are one-hot encoded. Numeric variables are median-imputed and scaled for models that benefit from scaling.

Missing predictor values are handled inside the modeling pipeline so the training and test sets are treated consistently. This matters in particular for Version B, where the SAQ features have skip-pattern missingness.

In [ ]:
def split_feature_types(df, feature_cols):
    categorical_cols = []
    numeric_cols = []

    for col in feature_cols:
        if df[col].dtype == "object":
            categorical_cols.append(col)
        else:
            unique_count = df[col].nunique(dropna=True)

            if unique_count <= 15:
                categorical_cols.append(col)
            else:
                numeric_cols.append(col)

    return numeric_cols, categorical_cols


def build_preprocessor(df, feature_cols):
    numeric_cols, categorical_cols = split_feature_types(df, feature_cols)

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, numeric_cols),
            ("categorical", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )

    return preprocessor, numeric_cols, categorical_cols

## Model set

The first model set covers a useful range of capacity for tabular regression on a heavy-tailed dollar target.

Linear Regression is the simplest baseline.
Elastic Net adds L1+L2 regularization for stability when features are correlated.
Random Forest captures non-linearities and interactions without manual feature engineering.
Gradient Boosting provides a stronger tabular regression benchmark with shallow trees and a slow learning rate.

There is no class-weighting in regression. The skew-handling layer comes in Objective 3, which retrains the same architectures on `log1p(TOTEXP23)`.

In [ ]:
def get_regression_models():
    models = {
        "linear_regression": LinearRegression(),
        "elastic_net": ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=42,
        ),
        "random_forest": RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42,
        ),
        "gradient_boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        ),
    }

    return models

## Evaluation function

The evaluation table reports MAE, RMSE, and R². For an outcome with a heavy right tail, MAE is the most stable single number to read off — it is in dollars and is not dominated by a small number of very-high-cost cases the way RMSE can be. RMSE is included because it penalizes large errors, which matters when high-cost cases are present. R² gives variance explained on the test set.

In [ ]:
def evaluate_regressor(model, X_test, y_test):
    pred = model.predict(X_test)

    metrics = {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_test, pred))),
        "R2": r2_score(y_test, pred),
    }

    return metrics

## Objective 1: Best predictive model

The first objective compares models based on overall predictive performance on the dollar scale. Because the target is right-skewed, RMSE and MAE are reported alongside R² so the trade-off between large errors on the high-cost tail and average dollar error is visible.

This objective tests two feature versions: adult features only (Version A), then adult features plus SAQ mental-health features (Version B).

In [ ]:
model_results = []
trained_models = {}

for version_name, feature_cols in model_versions.items():
    print("\n" + "=" * 100)
    print(version_name)
    print("=" * 100)

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]

    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(
        train_df,
        feature_cols
    )

    print("Numeric features:    ", len(numeric_cols))
    print("Categorical features:", len(categorical_cols))

    models = get_regression_models()

    for model_name, reg in models.items():
        print(f"\nTraining {model_name}...")

        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", reg),
        ])

        pipe.fit(X_train, y_train)

        metrics = evaluate_regressor(pipe, X_test, y_test)

        # Add the held-out high-cost MAE as a supporting check.
        pred_test = pipe.predict(X_test)
        if high_cost_mask_test.any():
            high_cost_mae = mean_absolute_error(
                y_test[high_cost_mask_test], pred_test[high_cost_mask_test]
            )
        else:
            high_cost_mae = float("nan")

        result = {
            "version": version_name,
            "model": model_name,
            "n_features": len(feature_cols),
            **metrics,
            "high_cost_MAE": high_cost_mae,
        }

        model_results.append(result)
        trained_models[(version_name, model_name)] = pipe

results_df = pd.DataFrame(model_results)

results_df = results_df.sort_values(
    ["RMSE", "MAE"],
    ascending=True
).reset_index(drop=True)

results_df


A_baseline_adult
Numeric features:     2
Categorical features: 21

Training linear_regression...

Training elastic_net...

Training random_forest...

Training gradient_boosting...

B_baseline_plus_saq
Numeric features:     3
Categorical features: 23

Training linear_regression...

Training elastic_net...

Training random_forest...

Training gradient_boosting...


,version,model,n_features,MAE,RMSE,R2,high_cost_MAE
0,A_baseline_adult,linear_regression,23,9526.773041,19410.552301,0.156857,39562.920293
1,B_baseline_plus_saq,linear_regression,26,9548.377292,19435.295515,0.154706,39518.660085
2,A_baseline_adult,elastic_net,23,9501.098639,19473.437585,0.151385,40411.927547
3,B_baseline_plus_saq,elastic_net,26,9501.081661,19484.210089,0.150446,40342.289226
4,A_baseline_adult,gradient_boosting,23,9381.742213,19719.355846,0.129816,40366.398422
5,B_baseline_plus_saq,random_forest,26,9538.472522,19740.039096,0.127990,39772.117759
6,A_baseline_adult,random_forest,23,9577.047790,19790.423137,0.123533,39892.703216
7,B_baseline_plus_saq,gradient_boosting,26,9417.668152,19903.063212,0.113527,40777.487658


## Objective 1 model comparison

The comparison table shows whether the added SAQ mental-health features improve predictive performance. If performance is nearly the same, the simpler adult-only model is preferred because it is easier to explain and uses the broader adult sample without SAQ-driven imputation.

In [ ]:
display_cols = [
    "version",
    "model",
    "n_features",
    "MAE",
    "RMSE",
    "R2",
    "high_cost_MAE",
]

results_df[display_cols]

,version,model,n_features,MAE,RMSE,R2,high_cost_MAE
0,A_baseline_adult,linear_regression,23,9526.773041,19410.552301,0.156857,39562.920293
1,B_baseline_plus_saq,linear_regression,26,9548.377292,19435.295515,0.154706,39518.660085
2,A_baseline_adult,elastic_net,23,9501.098639,19473.437585,0.151385,40411.927547
3,B_baseline_plus_saq,elastic_net,26,9501.081661,19484.210089,0.150446,40342.289226
4,A_baseline_adult,gradient_boosting,23,9381.742213,19719.355846,0.129816,40366.398422
5,B_baseline_plus_saq,random_forest,26,9538.472522,19740.039096,0.127990,39772.117759
6,A_baseline_adult,random_forest,23,9577.047790,19790.423137,0.123533,39892.703216
7,B_baseline_plus_saq,gradient_boosting,26,9417.668152,19903.063212,0.113527,40777.487658


## Objective 1 recommended model

The first recommended model is selected by RMSE on the dollar scale, with MAE and R² used to check whether the result is reasonable for a right-skewed cost target. The high-cost MAE is reported as a supporting metric — a model with low overall RMSE but very poor high-cost MAE may not be the right pick for a project that cares about expensive cases.

In [ ]:
best_row = results_df.iloc[0]

best_version = best_row["version"]
best_model_name = best_row["model"]
best_model = trained_models[(best_version, best_model_name)]

print("Best version:", best_version)
print("Best model:  ", best_model_name)
print(best_row[display_cols])

Best version: A_baseline_adult
Best model:   linear_regression
version           A_baseline_adult
model            linear_regression
n_features                      23
MAE                    9526.773041
RMSE                  19410.552301
R2                        0.156857
high_cost_MAE         39562.920293
Name: 0, dtype: object


## Objective 1 predicted-actual quintile review

The cross-tabulation shows how many adults in the held-out test set fall in each combination of actual-spend quintile and predicted-spend quintile. A model that predicts well will have most mass on the diagonal. This is the regression analog of a classification confusion matrix.

In [ ]:
best_features = model_versions[best_version]

X_test_best = test_df[best_features]
y_test = test_df[target_col]

best_pred = best_model.predict(X_test_best)

quintile_labels = ["Q1 (lowest)", "Q2", "Q3", "Q4", "Q5 (highest)"]

actual_quintile = pd.qcut(
    y_test, q=5, labels=quintile_labels, duplicates="drop"
)
predicted_quintile = pd.qcut(
    pd.Series(best_pred, index=y_test.index),
    q=5,
    labels=quintile_labels,
    duplicates="drop",
)

quintile_table = pd.crosstab(
    actual_quintile,
    predicted_quintile,
    rownames=["Actual quintile"],
    colnames=["Predicted quintile"],
)

quintile_table

Predicted quintile,Q1 (lowest),Q2,Q3,Q4,Q5 (highest)
Actual quintile,,,,,
Q1 (lowest),383,214,97,47,16
Q2,192,215,203,111,35
Q3,83,172,197,191,113
Q4,75,91,158,217,215
Q5 (highest),24,64,101,190,377


In [ ]:
# By-quintile error breakdown for the best model.
quintile_error = pd.DataFrame({
    "actual_quintile": actual_quintile,
    "actual": y_test.values,
    "predicted": best_pred,
    "abs_error": np.abs(y_test.values - best_pred),
})

quintile_error_summary = (
    quintile_error
    .groupby("actual_quintile", observed=True)
    .agg(
        n=("actual", "size"),
        mean_actual=("actual", "mean"),
        mean_predicted=("predicted", "mean"),
        MAE=("abs_error", "mean"),
    )
    .round(2)
)

quintile_error_summary

,n,mean_actual,mean_predicted,MAE
actual_quintile,,,,
Q1 (lowest),757,33.83,3220.31,4202.39
Q2,756,690.35,5975.90,5708.04
Q3,756,2507.35,9148.32,7005.37
Q4,756,7057.01,11958.42,6971.61
Q5 (highest),756,37216.21,16444.52,23753.50


## Objective 1 high-cost segment review

The high-cost segment review reports MAE and RMSE for adults whose actual `TOTEXP23` falls in the top 10%, top 20%, and top 30% of the test set. For a project that cares about expensive cases, this is more useful than a single overall metric.

A model that does well overall but poorly on the high-cost segments is fine for descriptive use but not for risk-management style use. A model that does well on the high-cost segments is the better candidate for outreach or screening.

In [ ]:
segment_rows = []

for top_pct in [0.10, 0.20, 0.30]:
    threshold = float(np.quantile(y_test, 1.0 - top_pct))
    mask = y_test >= threshold

    if mask.any():
        seg_mae = mean_absolute_error(y_test[mask], best_pred[mask])
        seg_rmse = float(np.sqrt(mean_squared_error(y_test[mask], best_pred[mask])))
    else:
        seg_mae = float("nan")
        seg_rmse = float("nan")

    segment_rows.append({
        "segment": f"Top {int(top_pct * 100)}% high-cost adults",
        "threshold": threshold,
        "n": int(mask.sum()),
        "MAE": seg_mae,
        "RMSE": seg_rmse,
    })

high_cost_segment_df = pd.DataFrame(segment_rows)

high_cost_segment_df

,segment,threshold,n,MAE,RMSE
0,Top 10% high-cost adults,24412.0,379,39562.920293,55920.329379
1,Top 20% high-cost adults,11621.0,757,23748.058043,40168.328939
2,Top 30% high-cost adults,6756.0,1135,18132.192731,33262.837013


## Objective 1 interpretation

The strongest first benchmark is the model that minimized RMSE on the held-out test set. RMSE is the right selection metric for this objective because the high-cost tail is part of the use case, and RMSE penalizes large prediction errors more than MAE does.

If the SAQ version (Version B) does not materially improve overall RMSE or high-cost MAE, the simpler adult-only model is the better candidate for reporting because it uses the broader adult population and avoids SAQ-driven imputation. If the SAQ version does improve high-cost MAE meaningfully, the trade-off between scope (broader population) and precision (better tail handling) is a project-level decision that the report should call out explicitly.

## Objective 1 feature importance

Feature importance is reviewed for the best model when it is a tree-based model. These values show which encoded features contributed most to prediction in the model. They should be interpreted as model signals, not causal effects.

In [ ]:
def get_feature_names_from_pipeline(pipeline):
    preprocessor = pipeline.named_steps["preprocessor"]
    return preprocessor.get_feature_names_out()


model_step = best_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    feature_names = get_feature_names_from_pipeline(best_model)

    importances = pd.DataFrame({
        "feature": feature_names,
        "importance": model_step.feature_importances_,
    })

    importances = importances.sort_values("importance", ascending=False).reset_index(drop=True)

    print(importances.head(25))
elif hasattr(model_step, "coef_"):
    feature_names = get_feature_names_from_pipeline(best_model)

    importances = pd.DataFrame({
        "feature": feature_names,
        "coef": model_step.coef_,
    })
    importances["abs_coef"] = importances["coef"].abs()
    importances = importances.sort_values("abs_coef", ascending=False).reset_index(drop=True)

    print("Best model is a linear model. Showing top coefficients by absolute value:")
    print(importances.head(25))
else:
    importances = pd.DataFrame()
    print(
        f"{best_model_name} does not expose feature_importances_ or coef_. "
        "Skipping the importance table."
    )

Best model is a linear model. Showing top coefficients by absolute value:
                                      feature          coef      abs_coef
0      categorical__chronic_condition_count_7  19844.057295  19844.057295
1    categorical__physical_difficulty_count_6  18200.796500  18200.796500
2      categorical__chronic_condition_count_9 -11336.448300  11336.448300
3    categorical__physical_difficulty_count_0  -9286.378530   9286.378530
4                   categorical__RTHLTH42_5.0   9247.940081   9247.940081
5                     categorical__INSC1231_2   8196.440257   8196.440257
6                     categorical__INSC1231_1  -8196.440257   8196.440257
7               categorical__MARRY23X_Unknown  -7670.560521   7670.560521
8      categorical__chronic_condition_count_0  -7480.870093   7480.870093
9    categorical__physical_difficulty_count_1  -7102.824537   7102.824537
10     categorical__chronic_condition_count_6   6258.283867   6258.283867
11     categorical__chronic_condition_

In [ ]:
importance_path = DATA_PROCESSED / "meps_objective1_feature_importance.csv"

if not importances.empty:
    importances.to_csv(importance_path, index=False)
    print("Saved:", importance_path)
else:
    print("No importance/coefficient table to save for the best model.")

Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective1_feature_importance.csv


## Save Objective 1 results

The Objective 1 comparison table, predicted-actual quintile review, high-cost segment review, and feature importance table are saved for the report and presentation.

In [ ]:
results_path = DATA_PROCESSED / "meps_objective1_model_comparison_results.csv"
quintile_path = DATA_PROCESSED / "meps_objective1_predicted_actual_quintile.csv"
segment_path = DATA_PROCESSED / "meps_objective1_high_cost_segment_review.csv"

results_df.to_csv(results_path, index=False)
quintile_table.to_csv(quintile_path)
high_cost_segment_df.to_csv(segment_path, index=False)

print("Saved:", results_path)
print("Saved:", quintile_path)
print("Saved:", segment_path)
if not importances.empty:
    print("Saved:", importance_path)

Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective1_model_comparison_results.csv
Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective1_predicted_actual_quintile.csv
Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective1_high_cost_segment_review.csv
Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective1_feature_importance.csv


## Objective 2: Best interpretable model

The second objective focuses on interpretability. The goal is to identify a model that can be explained clearly to technical and nontechnical readers while still performing reasonably well on the held-out test set.

This objective uses the simpler adult-only feature set from Version A. The SAQ layer is not used here because Objective 1 either showed it did not materially improve performance, or the added imputation reduces interpretability.

The main evaluation criteria are MAE and explanation quality. A model does not need to be the best-performing model overall to be useful for this objective. It should be understandable enough to support a clear project recommendation.

In [ ]:
objective2_features = model_versions["A_baseline_adult"]

X_train_obj2 = train_df[objective2_features]
y_train_obj2 = train_df[target_col]

X_test_obj2 = test_df[objective2_features]
y_test_obj2 = test_df[target_col]

preprocessor_obj2, numeric_cols_obj2, categorical_cols_obj2 = build_preprocessor(
    train_df,
    objective2_features
)

print("Objective 2 features:    ", len(objective2_features))
print("Numeric features:    ", numeric_cols_obj2)
print("Categorical features:", categorical_cols_obj2)

Objective 2 features:     23
Numeric features:     ['AGE23X', 'EDUCYR']
Categorical features: ['SEX', 'RACETHX', 'HISPANX', 'MARRY23X', 'REGION23', 'HIDEG', 'BORNUSA', 'FAMSZEYR', 'POVCAT23', 'FOODST23', 'RTHLTH42', 'MNHLTH42', 'chronic_condition_count', 'ANYLMI23', 'limitation_indicator_count', 'physical_difficulty_count', 'INSC1231', 'HAVEUS42', 'OFFER31X', 'HELD31X', 'access_barrier_count']


## Objective 2 model set

The interpretable model set uses simpler or more constrained members of the regression family. This keeps the comparison consistent with Objective 1 while changing the priority from maximum prediction to explanation.

The models are:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Shallow Regression Tree

The three linear models support coefficient-based explanation. Lasso additionally produces a sparse model: weak coefficients are shrunk to exactly zero, which makes the resulting story shorter to tell. The shallow tree supports rule-based explanation.

In [ ]:
objective2_models = {
    "linear_regression": LinearRegression(),
    "ridge_regression": Ridge(
        alpha=1.0,
        random_state=42,
    ),
    "lasso_regression": Lasso(
        alpha=1.0,
        max_iter=10000,
        random_state=42,
    ),
    "shallow_regression_tree": DecisionTreeRegressor(
        max_depth=4,
        min_samples_leaf=20,
        random_state=42,
    ),
}

## Train and evaluate Objective 2 models

Each model is trained on the same training data and evaluated on the same held-out test data. The comparison keeps the metrics from Objective 1 but adds an interpretation note for each model family.

In [ ]:
objective2_results = []
objective2_trained_models = {}

for model_name, reg in objective2_models.items():
    print(f"Training {model_name}...")

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_obj2),
        ("model", reg),
    ])

    pipe.fit(X_train_obj2, y_train_obj2)

    metrics = evaluate_regressor(pipe, X_test_obj2, y_test_obj2)
    pred_test = pipe.predict(X_test_obj2)
    if high_cost_mask_test.any():
        high_cost_mae = mean_absolute_error(
            y_test_obj2[high_cost_mask_test], pred_test[high_cost_mask_test]
        )
    else:
        high_cost_mae = float("nan")

    objective2_results.append({
        "objective": "Objective 2: interpretable model",
        "model": model_name,
        "n_features": len(objective2_features),
        **metrics,
        "high_cost_MAE": high_cost_mae,
    })

    objective2_trained_models[model_name] = pipe

objective2_results_df = pd.DataFrame(objective2_results)

objective2_results_df = objective2_results_df.sort_values(
    ["MAE", "RMSE"],
    ascending=True
).reset_index(drop=True)

objective2_results_df[[
    "objective",
    "model",
    "n_features",
    "MAE",
    "RMSE",
    "R2",
    "high_cost_MAE",
]]

Training linear_regression...
Training ridge_regression...
Training lasso_regression...
Training shallow_regression_tree...


,objective,model,n_features,MAE,RMSE,R2,high_cost_MAE
0,Objective 2: interpretable model,lasso_regression,23,9516.631073,19407.774400,0.157098,39573.343949
1,Objective 2: interpretable model,ridge_regression,23,9525.435410,19409.013307,0.156990,39567.834287
2,Objective 2: interpretable model,linear_regression,23,9526.773041,19410.552301,0.156857,39562.920293
3,Objective 2: interpretable model,shallow_regression_tree,23,9676.347902,19843.159706,0.118855,41992.492421


## Objective 2 recommendation

The recommended interpretable model is selected by MAE balanced with explanation quality. If two models are within a small MAE distance of each other, the more transparent model is preferred — for example, Lasso (sparse coefficients, easy to summarize as a short list of drivers) over Ridge with a 50-coefficient story, or a shallow tree if its split rules tell a cleaner story than a long coefficient list.

In [ ]:
objective2_best_row = objective2_results_df.iloc[0]
objective2_best_model_name = objective2_best_row["model"]
objective2_best_model = objective2_trained_models[objective2_best_model_name]

print("Objective 2 recommended model:", objective2_best_model_name)
print(objective2_best_row)

Objective 2 recommended model: lasso_regression
objective        Objective 2: interpretable model
model                            lasso_regression
n_features                                     23
MAE                                   9516.631073
RMSE                                   19407.7744
R2                                       0.157098
high_cost_MAE                        39573.343949
Name: 0, dtype: object


## Objective 2 explanation artifacts

Two artifacts are produced:

1. Top coefficients for the linear interpretable models (Linear, Ridge, Lasso). Because numeric features are standardized inside the preprocessor, coefficient magnitudes are roughly comparable across features within a single model. They are not directly comparable across models with different penalties — a Lasso coefficient of `0` means the feature was zeroed out by the L1 penalty, while a Linear coefficient of `0.001` just means the OLS estimate happened to be small.
2. The printed split rules of the shallow regression tree. Every leaf is a small, named segment of adults with an average predicted spend.

In [ ]:
def top_coefficients(pipe, top_n=15):
    pre = pipe.named_steps["preprocessor"]
    model = pipe.named_steps["model"]
    feature_names = pre.get_feature_names_out()
    coefs = model.coef_
    return (
        pd.DataFrame({"feature": feature_names, "coef": coefs})
        .assign(abs_coef=lambda d: d["coef"].abs())
        .sort_values("abs_coef", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


for model_name in ["linear_regression", "ridge_regression", "lasso_regression"]:
    print("\n" + "=" * 80)
    print(model_name + " — top coefficients")
    print("=" * 80)
    print(top_coefficients(objective2_trained_models[model_name], top_n=15))

# Lasso sparsity check.
lasso_pipe = objective2_trained_models["lasso_regression"]
lasso_coefs = lasso_pipe.named_steps["model"].coef_
lasso_zero = int((lasso_coefs == 0).sum())
lasso_total = len(lasso_coefs)
print(
    f"\nLasso sparsity: {lasso_zero} of {lasso_total} coefficients zeroed out "
    f"({lasso_zero / lasso_total:.0%})"
)


linear_regression — top coefficients
                                      feature          coef      abs_coef
0      categorical__chronic_condition_count_7  19844.057295  19844.057295
1    categorical__physical_difficulty_count_6  18200.796500  18200.796500
2      categorical__chronic_condition_count_9 -11336.448300  11336.448300
3    categorical__physical_difficulty_count_0  -9286.378530   9286.378530
4                   categorical__RTHLTH42_5.0   9247.940081   9247.940081
5                     categorical__INSC1231_2   8196.440257   8196.440257
6                     categorical__INSC1231_1  -8196.440257   8196.440257
7               categorical__MARRY23X_Unknown  -7670.560521   7670.560521
8      categorical__chronic_condition_count_0  -7480.870093   7480.870093
9    categorical__physical_difficulty_count_1  -7102.824537   7102.824537
10     categorical__chronic_condition_count_6   6258.283867   6258.283867
11     categorical__chronic_condition_count_5   6249.309921   6249.309921


In [ ]:
tree_pipe = objective2_trained_models["shallow_regression_tree"]
tree_pre = tree_pipe.named_steps["preprocessor"]
tree_model = tree_pipe.named_steps["model"]
tree_feature_names = list(tree_pre.get_feature_names_out())

print("Shallow regression tree split rules (max depth 4):")
print(export_text(tree_model, feature_names=tree_feature_names, max_depth=10))

Shallow regression tree split rules (max depth 4):
|--- categorical__limitation_indicator_count_0 <= 0.50
|   |--- categorical__RTHLTH42_5.0 <= 0.50
|   |   |--- categorical__chronic_condition_count_7 <= 0.50
|   |   |   |--- categorical__physical_difficulty_count_4 <= 0.50
|   |   |   |   |--- value: [17679.42]
|   |   |   |--- categorical__physical_difficulty_count_4 >  0.50
|   |   |   |   |--- value: [36962.75]
|   |   |--- categorical__chronic_condition_count_7 >  0.50
|   |   |   |--- value: [50351.63]
|   |--- categorical__RTHLTH42_5.0 >  0.50
|   |   |--- numeric__EDUCYR <= 0.69
|   |   |   |--- categorical__HIDEG_2.0 <= 0.50
|   |   |   |   |--- value: [27045.24]
|   |   |   |--- categorical__HIDEG_2.0 >  0.50
|   |   |   |   |--- value: [60159.72]
|   |   |--- numeric__EDUCYR >  0.69
|   |   |   |--- value: [70368.81]
|--- categorical__limitation_indicator_count_0 >  0.50
|   |--- categorical__chronic_condition_count_0 <= 0.50
|   |   |--- categorical__chronic_condition_count

## Save Objective 2 results

The Objective 2 model comparison table is saved for the report and presentation.

In [ ]:
objective2_results_path = DATA_PROCESSED / "meps_objective2_interpretable_model_results.csv"

objective2_results_df.to_csv(objective2_results_path, index=False)

print("Saved:", objective2_results_path)

Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective2_interpretable_model_results.csv


## Objective 3: Robust / skew-aware expenditure model

The third objective focuses on the heavy right tail of `TOTEXP23`. The same four model architectures from Objective 1 are retrained on `log1p(TOTEXP23)` instead of the raw dollar target so the loss is not dominated by a small number of very high-cost cases. Predictions are back-transformed with `expm1` and clipped at zero so they are reported on the dollar scale.

The metric focus is:

- MAE and RMSE on `log1p(TOTEXP23)` — the scale the model is actually optimizing.
- MAE and RMSE on the dollar scale (back-transformed) — apples-to-apples with Objective 1.
- MAE on the top-10% high-cost adults — the metric the project most cares about for risk-management style use cases.

In [ ]:
objective3_features = model_versions["A_baseline_adult"]

X_train_obj3 = train_df[objective3_features]
y_train_obj3 = train_df[target_log_col]

X_test_obj3 = test_df[objective3_features]
y_test_obj3_log = test_df[target_log_col]
y_test_obj3_dollars = test_df[target_col]

preprocessor_obj3, numeric_cols_obj3, categorical_cols_obj3 = build_preprocessor(
    train_df,
    objective3_features
)

print("Objective 3 features:", len(objective3_features))
print("Training target:     log1p(TOTEXP23)")
print("\nTraining log1p target summary:")
print(y_train_obj3.describe().round(3))

Objective 3 features: 23
Training target:     log1p(TOTEXP23)

Training log1p target summary:
count    11342.000
mean         6.943
std          3.202
min          0.000
25%          5.996
50%          7.809
75%          9.073
max         13.262
Name: TOTEXP23_log1p, dtype: float64


## Objective 3 helper functions

Two helpers support the dollar-scale and high-cost evaluation of log1p-trained models. The dollar-scale evaluator passes the log-space prediction through `expm1`, clips negative dollars at zero, and reports MAE / RMSE / R² on dollars. The high-cost capture rate measures the fraction of the held-out top-X% high-cost adults that the model also placed in its predicted top-X% — a regression analog of classification lift.

In [ ]:
def evaluate_log_to_dollars(model, X_test, y_test_dollars):
    pred_log = model.predict(X_test)
    pred_dollars = np.maximum(np.expm1(pred_log), 0.0)

    return pred_dollars, {
        "dollar_MAE": mean_absolute_error(y_test_dollars, pred_dollars),
        "dollar_RMSE": float(np.sqrt(mean_squared_error(y_test_dollars, pred_dollars))),
        "dollar_R2": r2_score(y_test_dollars, pred_dollars),
    }


def high_cost_capture_rate(y_true_dollars, y_pred_dollars, top_pct):
    n_top = int(np.ceil(len(y_true_dollars) * top_pct))

    actual_top = np.argsort(-y_true_dollars.values)[:n_top]
    pred_top = np.argsort(-y_pred_dollars)[:n_top]

    overlap = len(set(actual_top.tolist()) & set(pred_top.tolist()))
    return overlap / n_top

## Objective 3 model set

The four model architectures match Objective 1, but every model is trained on `log1p(TOTEXP23)`:

- Linear Regression on log1p target
- Elastic Net on log1p target
- Random Forest on log1p target
- Gradient Boosting on log1p target

In [ ]:
objective3_models = {
    "linear_regression_log1p": LinearRegression(),
    "elastic_net_log1p": ElasticNet(
        alpha=0.1,
        l1_ratio=0.5,
        max_iter=10000,
        random_state=42,
    ),
    "random_forest_log1p": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42,
    ),
    "gradient_boosting_log1p": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
}

## Train and evaluate Objective 3 models

Each model is evaluated with MAE and RMSE on the log1p scale, MAE / RMSE / R² on the back-transformed dollar scale, the dollar-scale MAE on the top-10% high-cost adults, and the capture rate of the top-20% high-cost adults.

In [ ]:
objective3_results = []
objective3_trained_models = {}

for model_name, reg in objective3_models.items():
    print(f"Training {model_name}...")

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_obj3),
        ("model", reg),
    ])

    pipe.fit(X_train_obj3, y_train_obj3)

    log_metrics = evaluate_regressor(pipe, X_test_obj3, y_test_obj3_log)
    pred_dollars, dollar_metrics = evaluate_log_to_dollars(
        pipe, X_test_obj3, y_test_obj3_dollars
    )

    if high_cost_mask_test.any():
        high_cost_mae = mean_absolute_error(
            y_test_obj3_dollars[high_cost_mask_test],
            pred_dollars[high_cost_mask_test],
        )
    else:
        high_cost_mae = float("nan")

    capture_top20 = high_cost_capture_rate(y_test_obj3_dollars, pred_dollars, 0.20)

    objective3_results.append({
        "objective": "Objective 3: robust / skew-aware",
        "model": model_name,
        "log_MAE": log_metrics["MAE"],
        "log_RMSE": log_metrics["RMSE"],
        "dollar_MAE": dollar_metrics["dollar_MAE"],
        "dollar_RMSE": dollar_metrics["dollar_RMSE"],
        "dollar_R2": dollar_metrics["dollar_R2"],
        "high_cost_MAE": high_cost_mae,
        "capture_rate_top20pct": capture_top20,
    })

    objective3_trained_models[model_name] = pipe

objective3_results_df = pd.DataFrame(objective3_results)

objective3_results_df = objective3_results_df.sort_values(
    ["high_cost_MAE", "dollar_MAE"],
    ascending=True,
).reset_index(drop=True)

objective3_results_df

Training linear_regression_log1p...
Training elastic_net_log1p...
Training random_forest_log1p...
Training gradient_boosting_log1p...


,objective,model,log_MAE,log_RMSE,dollar_MAE,dollar_RMSE,dollar_R2,high_cost_MAE,capture_rate_top20pct
0,Objective 3: robust / skew-aware,linear_regression_log1p,1.922302,2.527660,8236.784448,20934.388678,0.019277,47271.545676,0.454425
1,Objective 3: robust / skew-aware,gradient_boosting_log1p,1.842524,2.493336,7812.583617,20766.996749,0.034899,48779.787917,0.482166
2,Objective 3: robust / skew-aware,random_forest_log1p,1.885027,2.547151,7908.664914,20927.772244,0.019897,49686.745301,0.470277
3,Objective 3: robust / skew-aware,elastic_net_log1p,1.963894,2.597266,8051.869518,21315.235697,-0.016731,51335.334036,0.425363


## Objective 3 recommendation

The recommended model for this objective is selected by dollar-scale MAE on the top-10% high-cost adults, with overall dollar MAE and the top-20% capture rate used as supporting checks. This reflects a risk-management or outreach use case where the goal is to predict the most expensive adults well rather than the average adult.

In [ ]:
objective3_best_row = objective3_results_df.iloc[0]

objective3_best_model_name = objective3_best_row["model"]
objective3_best_model = objective3_trained_models[objective3_best_model_name]

print("Objective 3 recommended model:", objective3_best_model_name)
print(objective3_best_row)

Objective 3 recommended model: linear_regression_log1p
objective                Objective 3: robust / skew-aware
model                             linear_regression_log1p
log_MAE                                          1.922302
log_RMSE                                          2.52766
dollar_MAE                                    8236.784448
dollar_RMSE                                  20934.388678
dollar_R2                                        0.019277
high_cost_MAE                                47271.545676
capture_rate_top20pct                            0.454425
Name: 0, dtype: object


## Objective 3 interpretation

Training on the log1p target compresses the right tail and gives every model a more even loss surface to optimize. Whether this translates into better dollar-scale prediction on the high-cost segment depends on the model: for tree-based models the gain is usually modest (they are already non-linear), while for linear models the log1p fit can change the picture significantly because the linear-on-raw-dollars version is dominated by the few highest-cost cases.

The capture rate of the top-20% high-cost adults answers a slightly different question: not "how big are the dollar errors?" but "do we put the right adults at the top of the predicted ranking?" This is the metric to read for a screening or outreach use case.

## Objective 3 high-cost segment review

The high-cost segment review for the recommended Objective 3 model shows MAE and RMSE on the dollar scale at three high-cost cutoffs. This supports the cost-sensitive framing — false negatives and false positives may matter differently depending on the use case.

In [ ]:
objective3_pred_dollars, _ = evaluate_log_to_dollars(
    objective3_best_model,
    X_test_obj3,
    y_test_obj3_dollars,
)

objective3_segment_rows = []

for top_pct in [0.10, 0.20, 0.30]:
    threshold = float(np.quantile(y_test_obj3_dollars, 1.0 - top_pct))
    mask = y_test_obj3_dollars >= threshold

    if mask.any():
        seg_mae = mean_absolute_error(
            y_test_obj3_dollars[mask], objective3_pred_dollars[mask]
        )
        seg_rmse = float(np.sqrt(mean_squared_error(
            y_test_obj3_dollars[mask], objective3_pred_dollars[mask]
        )))
    else:
        seg_mae = float("nan")
        seg_rmse = float("nan")

    capture = high_cost_capture_rate(
        y_test_obj3_dollars, objective3_pred_dollars, top_pct
    )

    objective3_segment_rows.append({
        "segment": f"Top {int(top_pct * 100)}% high-cost adults",
        "threshold": threshold,
        "n": int(mask.sum()),
        "MAE": seg_mae,
        "RMSE": seg_rmse,
        "capture_rate": capture,
    })

objective3_segment_df = pd.DataFrame(objective3_segment_rows)

objective3_segment_df

,segment,threshold,n,MAE,RMSE,capture_rate
0,Top 10% high-cost adults,24412.0,379,47271.545676,62292.278406,0.321900
1,Top 20% high-cost adults,11621.0,757,30061.737896,45412.669998,0.454425
2,Top 30% high-cost adults,6756.0,1135,22480.600952,37494.760696,0.573568


## Save Objective 3 results

The Objective 3 comparison table and segment review are saved for the report and presentation.

In [ ]:
objective3_results_path = DATA_PROCESSED / "meps_objective3_robust_skew_aware_results.csv"
objective3_segment_path = DATA_PROCESSED / "meps_objective3_high_cost_segment_review.csv"

objective3_results_df.to_csv(objective3_results_path, index=False)
objective3_segment_df.to_csv(objective3_segment_path, index=False)

print("Saved:", objective3_results_path)
print("Saved:", objective3_segment_path)

Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective3_robust_skew_aware_results.csv
Saved: /content/drive/MyDrive/med_project/data/processed/meps_objective3_high_cost_segment_review.csv


## Modeling checkpoint

Part II now includes three regression objectives. Each objective trains and compares at least four candidate models, uses metrics tied to the objective, and recommends a model based on the project context.

The modeling results are saved for the report, presentation, and later scoring workflow.

In [ ]:
modeling_checkpoint = pd.DataFrame({
    "objective": [
        "Objective 1: Best predictive model",
        "Objective 2: Best interpretable model",
        "Objective 3: Robust / skew-aware",
    ],
    "models_trained": [
        len(results_df),
        len(objective2_results_df),
        len(objective3_results_df),
    ],
    "recommended_model": [
        f"{best_version} / {best_model_name}",
        objective2_best_model_name,
        objective3_best_model_name,
    ],
    "selection_metric": [
        "RMSE on dollar scale",
        "MAE plus explanation quality",
        "high-cost MAE on dollar scale",
    ],
})

modeling_checkpoint

,objective,models_trained,recommended_model,selection_metric
0,Objective 1: Best predictive model,8,A_baseline_adult / linear_regression,RMSE on dollar scale
1,Objective 2: Best interpretable model,4,lasso_regression,MAE plus explanation quality
2,Objective 3: Robust / skew-aware,4,linear_regression_log1p,high-cost MAE on dollar scale


## Save modeling checkpoint

The final checkpoint table is saved so the report can clearly show how Part II met the modeling requirements.

In [ ]:
checkpoint_path = DATA_PROCESSED / "meps_modeling_checkpoint.csv"

modeling_checkpoint.to_csv(checkpoint_path, index=False)

print("Saved:", checkpoint_path)

Saved: /content/drive/MyDrive/med_project/data/processed/meps_modeling_checkpoint.csv
